In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json
from pathlib import Path

# ── Charger les données nutritionnelles ──────────────────
csv_path     = Path("C:/SmartMeal/data/usda/nutrition_food10.csv")
df_nutrition = pd.read_csv(csv_path)

print("✅ Imports OK")
print(f"📊 Données chargées : {df_nutrition.shape}")
print(df_nutrition[["class_name","calories","protein_g","fat_g","carbs_g"]].to_string(index=False))

✅ Imports OK
📊 Données chargées : (10, 8)
    class_name  calories  protein_g  fat_g  carbs_g
         pizza       266       11.0   10.0     33.0
         sushi       143        5.8    3.2     21.0
     hamburger       295       17.0   14.0     24.0
  caesar_salad       190        8.0   15.0      8.0
         steak       271       26.0   18.0      0.0
     apple_pie       237        2.0   11.0     34.0
      omelette       154       10.0   12.0      1.0
grilled_salmon       208       20.0   13.0      0.0
     ice_cream       207        3.5   11.0     24.0
      bibimbap       490       26.0   12.0     69.0


In [2]:
def calculate_health_score(food_data):
    """
    Score de santé sur 100 basé sur les nutriments.
    Inspiré du Nutri-Score européen.
    """
    score = 50  # base neutre

    cal      = food_data.get("calories",   0)
    protein  = food_data.get("protein_g",  0)
    fat      = food_data.get("fat_g",      0)
    carbs    = food_data.get("carbs_g",    0)
    fiber    = food_data.get("fiber_g",    0)
    sugar    = food_data.get("sugar_g",    0)
    sodium   = food_data.get("sodium_mg",  0)

    # ── Points positifs ──────────────────────────────────
    # Protéines (bonnes pour la satiété et muscles)
    if protein >= 20 : score += 20
    elif protein >= 10: score += 10
    elif protein >= 5 : score += 5

    # Fibres (bonnes pour la digestion)
    if fiber >= 5  : score += 15
    elif fiber >= 3: score += 10
    elif fiber >= 1: score += 5

    # Calories modérées (idéal 150-300 kcal/100g)
    if 150 <= cal <= 300: score += 10

    # ── Points négatifs ──────────────────────────────────
    # Calories excessives
    if cal > 400  : score -= 20
    elif cal > 300: score -= 10

    # Graisses excessives
    if fat > 20  : score -= 15
    elif fat > 15: score -= 8

    # Sucres excessifs
    if sugar > 15 : score -= 15
    elif sugar > 8: score -= 8

    # Sodium excessif
    if sodium > 600 : score -= 20
    elif sodium > 400: score -= 10
    elif sodium > 200: score -= 5

    # ── Clamp entre 0 et 100 ─────────────────────────────
    score = max(0, min(100, score))
    return round(score)


def get_grade(score):
    if score >= 80: return "A", "🟢", "Excellent"
    if score >= 65: return "B", "🟡", "Bon"
    if score >= 50: return "C", "🟠", "Moyen"
    if score >= 35: return "D", "🔴", "Médiocre"
    return              "E", "⛔", "Mauvais"


# ── Calculer les scores ───────────────────────────────────
print("📊 Calcul des scores de santé...\n")
results = []

for _, row in df_nutrition.iterrows():
    score         = calculate_health_score(row.to_dict())
    grade, emoji, label = get_grade(score)
    results.append({
        "class_name" : row["class_name"],
        "score"      : score,
        "grade"      : grade,
        "emoji"      : emoji,
        "label"      : label
    })
    print(f"  {emoji} {row['class_name']:20s} → Score: {score:3d}/100  [{grade}] {label}")

df_scores = pd.DataFrame(results).sort_values("score", ascending=False)
print(f"\n🏆 Meilleur aliment  : {df_scores.iloc[0]['class_name']} ({df_scores.iloc[0]['score']}/100)")
print(f"⚠️  Pire aliment      : {df_scores.iloc[-1]['class_name']} ({df_scores.iloc[-1]['score']}/100)")

📊 Calcul des scores de santé...

  🟡 pizza                → Score:  65/100  [B] Bon
  🟠 sushi                → Score:  50/100  [C] Moyen
  🟡 hamburger            → Score:  70/100  [B] Bon
  🟠 caesar_salad         → Score:  60/100  [C] Moyen
  🟡 steak                → Score:  72/100  [B] Bon
  🟠 apple_pie            → Score:  50/100  [C] Moyen
  🟡 omelette             → Score:  65/100  [B] Bon
  🟢 grilled_salmon       → Score:  80/100  [A] Excellent
  🔴 ice_cream            → Score:  45/100  [D] Médiocre
  🔴 bibimbap             → Score:  40/100  [D] Médiocre

🏆 Meilleur aliment  : grilled_salmon (80/100)
⚠️  Pire aliment      : bibimbap (40/100)


In [3]:
# ── Règles de compatibilité digestive ────────────────────
# Sources : principes de chrono-nutrition et diététique
combinations = [
    # (aliment_1, aliment_2, compatible, raison)
    ("steak",          "caesar_salad",   True,  "Protéines + légumes verts = digestion optimale"),
    ("grilled_salmon", "caesar_salad",   True,  "Protéines légères + fibres = excellent"),
    ("omelette",       "caesar_salad",   True,  "Oeufs + légumes = combo équilibré"),
    ("sushi",          "caesar_salad",   True,  "Poisson + légumes = très digeste"),
    ("bibimbap",       "caesar_salad",   True,  "Riz + légumes = équilibré"),
    ("pizza",          "hamburger",      False, "Double glucides + lipides = lourdeur digestive"),
    ("apple_pie",      "ice_cream",      False, "Double sucres = pic glycémique"),
    ("hamburger",      "ice_cream",      False, "Lipides + sucres froids = digestion lente"),
    ("steak",          "apple_pie",      False, "Protéines + sucres = fermentation intestinale"),
    ("pizza",          "ice_cream",      False, "Glucides + sucres = surcharge pancréatique"),
    ("steak",          "hamburger",      False, "Double protéines lourdes = surcharge digestive"),
    ("grilled_salmon", "sushi",          True,  "Double poisson = digestion facile"),
    ("omelette",       "sushi",          True,  "Protéines légères compatibles"),
    ("bibimbap",       "omelette",       True,  "Riz + oeuf = combo traditionnel équilibré"),
    ("apple_pie",      "caesar_salad",   True,  "Sucres + fibres = index glycémique modéré"),
    ("pizza",          "caesar_salad",   False, "Glucides lourds + vinaigrette acide = ballonnements"),
    ("hamburger",      "caesar_salad",   True,  "Viande + légumes = acceptable"),
    ("ice_cream",      "grilled_salmon", False, "Sucres froids + protéines = digestion difficile"),
    ("steak",          "bibimbap",       False, "Protéines lourdes + riz = surcharge"),
    ("sushi",          "ice_cream",      False, "Poisson cru + sucres froids = risque digestif"),
]

df_combinations = pd.DataFrame(
    combinations,
    columns=["food_1", "food_2", "compatible", "reason"]
)

total_compat  = df_combinations["compatible"].sum()
total_incomp  = (~df_combinations["compatible"]).sum()

print(f"✅ Dataset combinaisons créé")
print(f"   Total paires     : {len(df_combinations)}")
print(f"   ✅ Compatibles   : {total_compat}")
print(f"   ❌ Incompatibles : {total_incomp}")
print(f"\n{df_combinations.to_string(index=False)}")

# Sauvegarder
combo_path = Path("C:/SmartMeal/data/combinations/food_combinations.csv")
combo_path.parent.mkdir(exist_ok=True)
df_combinations.to_csv(combo_path, index=False)
print(f"\n✅ Sauvegardé : {combo_path}")

✅ Dataset combinaisons créé
   Total paires     : 20
   ✅ Compatibles   : 10
   ❌ Incompatibles : 10

        food_1         food_2  compatible                                              reason
         steak   caesar_salad        True      Protéines + légumes verts = digestion optimale
grilled_salmon   caesar_salad        True              Protéines légères + fibres = excellent
      omelette   caesar_salad        True                   Oeufs + légumes = combo équilibré
         sushi   caesar_salad        True                    Poisson + légumes = très digeste
      bibimbap   caesar_salad        True                           Riz + légumes = équilibré
         pizza      hamburger       False      Double glucides + lipides = lourdeur digestive
     apple_pie      ice_cream       False                      Double sucres = pic glycémique
     hamburger      ice_cream       False           Lipides + sucres froids = digestion lente
         steak      apple_pie       False       Prot

In [4]:
scores_path = Path("C:/SmartMeal/data/usda/health_scores.csv")
df_scores.to_csv(scores_path, index=False)

print(f"✅ Scores sauvegardés : {scores_path}")
print(f"\n📊 Résumé final Phase 4 :")
print(f"{'─'*45}")
for _, row in df_scores.iterrows():
    print(f"  {row['emoji']} {row['class_name']:20s} {row['score']:3d}/100  {row['label']}")
print(f"{'─'*45}")
print(f"\n✅ Phase 4 terminée !")
print(f"   Score moyen     : {df_scores['score'].mean():.1f}/100")
print(f"   Combos créées   : {len(df_combinations)}")

✅ Scores sauvegardés : C:\SmartMeal\data\usda\health_scores.csv

📊 Résumé final Phase 4 :
─────────────────────────────────────────────
  🟢 grilled_salmon        80/100  Excellent
  🟡 steak                 72/100  Bon
  🟡 hamburger             70/100  Bon
  🟡 pizza                 65/100  Bon
  🟡 omelette              65/100  Bon
  🟠 caesar_salad          60/100  Moyen
  🟠 sushi                 50/100  Moyen
  🟠 apple_pie             50/100  Moyen
  🔴 ice_cream             45/100  Médiocre
  🔴 bibimbap              40/100  Médiocre
─────────────────────────────────────────────

✅ Phase 4 terminée !
   Score moyen     : 59.7/100
   Combos créées   : 20


In [2]:

import numpy as np
from sklearn.metrics import cohen_kappa_score, confusion_matrix
import pandas as pd

def calculate_inter_annotator_agreement(annotations_annotator1, annotations_annotator2):
    """Calculer l'accord inter-annotateurs"""
    
    # Supposons que annotations = [(food1, food2, compatibility), ...]
    # compatibility = 1 (bon) ou 0 (mauvais)
    
    pairs1 = [pair[2] for pair in annotations_annotator1]
    pairs2 = [pair[2] for pair in annotations_annotator2]
    
    # Cohen's Kappa
    kappa = cohen_kappa_score(pairs1, pairs2)
    
    # Simple accuracy
    agreement = np.mean(np.array(pairs1) == np.array(pairs2))
    
    print(f"\nInter-Annotator Agreement:")
    print(f"  Simple Agreement: {agreement:.2%}")
    print(f"  Cohen's Kappa:    {kappa:.4f}")
    
    if kappa > 0.8:
        print(f"  Interpretation:   EXCELLENT")
    elif kappa > 0.6:
        print(f"  Interpretation:   GOOD")
    elif kappa > 0.4:
        print(f"  Interpretation:   MODERATE")
    else:
        print(f"  Interpretation:   WEAK")
    
    return kappa, agreement

def visualize_compatibility_matrix(compatibility_data, class_names):
    """Matrice de compatibilité"""
    
    # compatibility_data = dict avec clé = (food1_idx, food2_idx), value = compatibility_score
    
    n_foods = len(class_names)
    compatibility_matrix = np.zeros((n_foods, n_foods))
    
    for (food1, food2), score in compatibility_data.items():
        compatibility_matrix[food1, food2] = score
        compatibility_matrix[food2, food1] = score  # Symétrique
    
    plt.figure(figsize=(14, 12))
    sns.heatmap(compatibility_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'label': 'Compatibility Score'})
    plt.title('Food Compatibility Matrix', fontsize=16, fontweight='bold')
    plt.xlabel('Food Item', fontsize=12)
    plt.ylabel('Food Item', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig('compatibility_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()

def worst_compatible_pairs(compatibility_data, class_names, top_k=10):
    """Les paires d'aliments les moins compatibles"""
    
    sorted_pairs = sorted(compatibility_data.items(), key=lambda x: x[1])
    
    print("\nTop Incompatible Food Pairs:")
    print(f"{'Rank':>5s} {'Food 1':20s} {'Food 2':20s} {'Score':>10s}")
    print("-" * 58)
    
    for rank, ((food1_idx, food2_idx), score) in enumerate(sorted_pairs[:top_k], 1):
        f1 = class_names[food1_idx] if food1_idx < len(class_names) else f"Food_{food1_idx}"
        f2 = class_names[food2_idx] if food2_idx < len(class_names) else f"Food_{food2_idx}"
        print(f"{rank:>5d} {f1:20s} {f2:20s} {score:>10.2f}")

